## 1. Install and import dependencies

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid")

print("pandas:", pd.__version__)
print("numpy:", np.__version__)

pandas: 3.0.6
numpy: 2.5.4


## 2. Configure multi-region input paths and target country output

In [2]:
INPUT_CSVS = [
    Path("../../data/SE1_AGGREGATED_GENERATION_PER_TYPE_GENERATION.csv"),
    Path("../../data/SE2_AGGREGATED_GENERATION_PER_TYPE_GENERATION.csv"),
    Path("../../data/SE3_AGGREGATED_GENERATION_PER_TYPE_GENERATION.csv"),
    Path("../../data/SE4_AGGREGATED_GENERATION_PER_TYPE_GENERATION.csv"),
]

TARGET_COUNTRY_NAME = "Sweden"  # Or string code representing the target country
OUTPUT_CSV = Path("../../data/SE_generation_with_estimated_carbon_intensity.csv")

INTERVAL_HOURS = 0.25

missing_files = [p for p in INPUT_CSVS if not p.exists()]
if missing_files:
    print(f"Warning: The following input files were not found: {missing_files}")
else:
    print(f"Found {len(INPUT_CSVS)} input region files.")
print("Output file:", OUTPUT_CSV.resolve())

Output file: C:\University25\master\_teza_master\soft\project\data\data\SE_generation_with_estimated_carbon_intensity.csv


## 3. Load, combine, and inspect the source data from all regions

In [3]:
df_list = []
required_columns = {"MTU (UTC)", "Area", "Production Type", "Generation (MW)"}

for file_path in INPUT_CSVS:
    if file_path.exists():
        region_df = pd.read_csv(file_path, dtype={"Generation (MW)": "string"})
        missing_columns = required_columns - set(region_df.columns)
        if missing_columns:
            raise ValueError(f"File {file_path} is missing required columns: {sorted(missing_columns)}")
        region_df["Source_File"] = file_path.name
        df_list.append(region_df)

if df_list:
    df = pd.concat(df_list, ignore_index=True)
    print("Total Rows Loaded:", len(df))
    print("Regions / Areas found:", df["Area"].unique().tolist())
else:
    print("No input data loaded. Check file paths.")

No input data loaded. Check file paths.


## 4. Define lifecycle emission factors

In [4]:
EMISSION_FACTORS = {
    "Wind Onshore": 11.0,
    "Wind Offshore": 11.0,
    "Solar": 48.0,
    "Nuclear": 12.0,
    "Hydro Run-of-river and pondage": 24.0,
    "Hydro Water Reservoir": 24.0,
    "Biomass": 230.0,
    "Fossil Gas": 490.0,
    "Fossil Coal-derived gas": 490.0,
    "Fossil Brown coal/Lignite": 820.0,
    "Fossil Hard coal": 820.0,
    "Fossil Oil": 650.0,
    "Fossil Oil shale": 650.0,
    "Fossil Peat": 820.0,
    "Geothermal": 24.0,
    "Marine": 24.0,
    "Other renewable": 48.0,
    "Other": 490.0,
    "Waste": 500.0,
}

EXCLUDED_SOURCES = {
    "Energy storage",
    "Hydro Pumped Storage",
}

## 5. Clean generation values, convert MTU to CET/CEST, and parse interval timestamps

In [5]:
work = df.copy()

# Convert generation values to numeric
work["Generation (MW)"] = pd.to_numeric(
    work["Generation (MW)"].astype("string").str.strip().replace({"": pd.NA, "n/e": pd.NA, "-": pd.NA}),
    errors="coerce"
)

# Helper function to convert UTC interval strings to CET/CEST interval strings
def utc_interval_to_cet(mtu_utc_str):
    if pd.isna(mtu_utc_str):
        return pd.NA
    parts = str(mtu_utc_str).split("-")
    if len(parts) != 2:
        return mtu_utc_str
    try:
        start_utc = pd.to_datetime(parts[0].strip(), dayfirst=True)
        end_utc = pd.to_datetime(parts[1].strip(), dayfirst=True)
        # Convert UTC to Europe/Berlin (CET/CEST)
        start_cet = start_utc.tz_localize("UTC").tz_convert("Europe/Berlin")
        end_cet = end_utc.tz_localize("UTC").tz_convert("Europe/Berlin")
        fmt = "%d.%m.%Y %H:%M"
        return f"{start_cet.strftime(fmt)} - {end_cet.strftime(fmt)}"
    except Exception:
        return mtu_utc_str

# Map MTU (UTC) to MTU (CET/CEST)
work["MTU (CET/CEST)"] = work["MTU (UTC)"].apply(utc_interval_to_cet)

# Split interval strings for internal datetime sorting/filtering
interval_parts = work["MTU (UTC)"].astype("string").str.split(r"\s+-\s+", n=1, expand=True)
work["Interval start"] = pd.to_datetime(interval_parts[0], dayfirst=True, errors="coerce")
work["Interval end"] = pd.to_datetime(interval_parts[1], dayfirst=True, errors="coerce")

work["Lifecycle emission factor (gCO2eq/kWh)"] = work["Production Type"].map(EMISSION_FACTORS)

work["Included in carbon-intensity calculation"] = (
    work["Lifecycle emission factor (gCO2eq/kWh)"].notna()
    & work["Generation (MW)"].notna()
)

print("Rows processed:", len(work))
display(work[["MTU (UTC)", "MTU (CET/CEST)", "Area", "Production Type", "Generation (MW)"]].head())

NameError: name 'df' is not defined

## 6. Calculate emissions per source row across all regions

In [ ]:
work["Generation included (MW)"] = work["Generation (MW)"].where(
    work["Included in carbon-intensity calculation"], 0.0
)

work["Estimated emissions (tCO2eq per 15 min)"] = (
    work["Generation (MW)"]
    * INTERVAL_HOURS
    * work["Lifecycle emission factor (gCO2eq/kWh)"]
    / 1000.0
)

work.loc[~work["Included in carbon-intensity calculation"],
         "Estimated emissions (tCO2eq per 15 min)"] = np.nan

## 7. Aggregate generation and compute target country average CI per interval

In [ ]:
work["Weighted generation x factor"] = (
    work["Generation included (MW)"]
    * work["Lifecycle emission factor (gCO2eq/kWh)"].fillna(0.0)
)

COUNTRY_GROUP_KEYS = ["MTU (UTC)"]

interval_summary = (
    work.groupby(COUNTRY_GROUP_KEYS, dropna=False)
    .agg(
        **{
            "Total generation included (MW)": ("Generation included (MW)", "sum"),
            "Weighted generation x factor": ("Weighted generation x factor", "sum"),
            "Estimated total emissions (tCO2eq per 15 min)": (
                "Estimated emissions (tCO2eq per 15 min)",
                lambda s: s.sum(min_count=1)
            ),
        }
    )
    .reset_index()
)

interval_summary["Estimated carbon intensity (gCO2eq/kWh)"] = np.where(
    interval_summary["Total generation included (MW)"] > 0,
    interval_summary["Weighted generation x factor"]
        / interval_summary["Total generation included (MW)"],
    np.nan
)

interval_summary = interval_summary.drop(columns=["Weighted generation x factor"])
print("Total Intervals Aggregated:", len(interval_summary))

## 8. Join metrics back and export with requested column names

In [ ]:
metrics = interval_summary[COUNTRY_GROUP_KEYS + [
    "Estimated carbon intensity (gCO2eq/kWh)",
    "Total generation included (MW)",
    "Estimated total emissions (tCO2eq per 15 min)"
]]

work = work.merge(metrics, on=COUNTRY_GROUP_KEYS, how="left")

# Exact column name order requested by user:
# 1. MTU (CET/CEST)
# 2. Area
# 3. Production Type
# 4. Generation (MW)
# 5. Lifecycle emission factor (gCO2eq/kWh)
# 6. Estimated emissions (tCO2eq per 15 min)
# 7. Estimated carbon intensity (gCO2eq/kWh)
# 8. Total generation included (MW)
# 9. Estimated total emissions (tCO2eq per 15 min)

output_columns = [
    "MTU (CET/CEST)",
    "Area",
    "Production Type",
    "Generation (MW)",
    "Lifecycle emission factor (gCO2eq/kWh)",
    "Estimated emissions (tCO2eq per 15 min)",
    "Estimated carbon intensity (gCO2eq/kWh)",
    "Total generation included (MW)",
    "Estimated total emissions (tCO2eq per 15 min)"
]

result = work[output_columns].copy()
result.to_csv(OUTPUT_CSV, index=False)
print(f"Successfully saved {len(result):,} rows to: {OUTPUT_CSV.resolve()}")
display(result.head(10))